In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/O.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed2026_a_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed123_d_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/M.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed3407_b_raw.csv
/kaggle/input/datasets/mohankrishnathalla/pr

## Setup and Data Loading

In [2]:
import pandas as pd
import numpy as np
 
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
DS   = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/'
 
sub = pd.read_csv(COMP + 'sample_submission.csv')
 
a    = pd.read_csv(DS + 'A.csv').rename(columns={'Irrigation_Need':'A'})
b    = pd.read_csv(DS + 'B.csv').rename(columns={'Irrigation_Need':'B'})
c    = pd.read_csv(DS + 'C.csv').rename(columns={'Irrigation_Need':'C'})
d    = pd.read_csv(DS + 'D.csv').rename(columns={'Irrigation_Need':'D'})
x    = pd.read_csv(DS + 'X.csv').rename(columns={'Irrigation_Need':'X'})
m    = pd.read_csv(DS + 'M.csv').rename(columns={'Irrigation_Need':'M'})
n    = pd.read_csv(DS + 'N.csv').rename(columns={'Irrigation_Need':'N'})
o    = pd.read_csv(DS + 'O.csv').rename(columns={'Irrigation_Need':'O'})
p    = pd.read_csv(DS + 'P.csv').rename(columns={'Irrigation_Need':'P'})
ours = pd.read_csv(DS + 'submission_d4_s999.csv').rename(columns={'Irrigation_Need':'OURS'})
 
dfs = (a.merge(b,on='id').merge(c,on='id').merge(d,on='id')
        .merge(x,on='id').merge(m,on='id').merge(n,on='id')
        .merge(o,on='id').merge(p,on='id').merge(ours,on='id'))
 
# Transfer predictions (base)
dfs['abcd_agree'] = (dfs['A']==dfs['B']) & (dfs['B']==dfs['C']) & (dfs['C']==dfs['D'])
dfs['transfer']   = dfs.apply(lambda r: r['M'] if r['abcd_agree'] else r['X'], axis=1)
 
print(f"ABCD agree: {dfs['abcd_agree'].sum():,}  disagree: {(~dfs['abcd_agree']).sum():,}")
 
SLICES = [125000, 130000, 135000, 137400, 140000, 145000]
 
for sl in SLICES:
    final = []
    for i in range(len(dfs)):
        if i < sl:
            final.append(dfs['N'].iloc[i])
        else:
            if dfs['N'].iloc[i] == dfs['O'].iloc[i]:
                final.append(dfs['N'].iloc[i])
            else:
                final.append(dfs['P'].iloc[i])
 
    out = sub.copy()
    out['Irrigation_Need'] = final
    fname = f'submission_slice_{sl}.csv'
    out.to_csv(fname, index=False)
    dist = out['Irrigation_Need'].value_counts().to_dict()
    print(f"slice={sl:,}  dist={dist}  → {fname}")
 
for sl in [135000, 137400, 140000]:
    final = []
    for i in range(len(dfs)):
        if i < sl:
            final.append(dfs['N'].iloc[i])
        else:
            if dfs['N'].iloc[i] == dfs['O'].iloc[i]:
                final.append(dfs['N'].iloc[i])
            else:
                final.append(dfs['OURS'].iloc[i]) 
 
    out = sub.copy()
    out['Irrigation_Need'] = final
    fname = f'submission_slice_{sl}_ours_aux.csv'
    out.to_csv(fname, index=False)
    dist = out['Irrigation_Need'].value_counts().to_dict()
    print(f"slice={sl:,} ours_aux dist={dist}  → {fname}")
 
print("\n" + "="*55)

ABCD agree: 269,295  disagree: 705
slice=125,000  dist={'Low': 159501, 'Medium': 100308, 'High': 10191}  → submission_slice_125000.csv
slice=130,000  dist={'Low': 159500, 'Medium': 100308, 'High': 10192}  → submission_slice_130000.csv
slice=135,000  dist={'Low': 159500, 'Medium': 100306, 'High': 10194}  → submission_slice_135000.csv
slice=137,400  dist={'Low': 159499, 'Medium': 100304, 'High': 10197}  → submission_slice_137400.csv
slice=140,000  dist={'Low': 159499, 'Medium': 100302, 'High': 10199}  → submission_slice_140000.csv
slice=145,000  dist={'Low': 159498, 'Medium': 100302, 'High': 10200}  → submission_slice_145000.csv
slice=135,000 ours_aux dist={'Low': 159485, 'Medium': 100286, 'High': 10229}  → submission_slice_135000_ours_aux.csv
slice=137,400 ours_aux dist={'Low': 159484, 'Medium': 100287, 'High': 10229}  → submission_slice_137400_ours_aux.csv
slice=140,000 ours_aux dist={'Low': 159484, 'Medium': 100287, 'High': 10229}  → submission_slice_140000_ours_aux.csv

